In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -U transformers

In [ ]:
import json
import torch
import numpy as np
import random
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel, set_seed
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
MODELS = [
    "dbmdz/bert-base-italian-xxl-cased",
    "linhd-postdata/alberti-bert-base-multilingual-cased",
    "mattiaferrarini/saba"
]

In [ ]:
FILE_PATHS = [
    "/content/drive/My Drive/italian_poems/italian_rhymes_dict_10.json",
    "/content/drive/My Drive/italian_poems/italian_rhymes_dict_25.json",
    "/content/drive/My Drive/italian_poems/italian_rhymes_dict_50.json",
    "/content/drive/My Drive/italian_poems/italian_rhymes_dict_75.json",
    "/content/drive/My Drive/italian_poems/italian_rhymes_dict_100.json",
    "/content/drive/My Drive/italian_poems/italian_rhymes_dict_125.json"
]

In [ ]:
BATCH_SIZE = 32
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def fix_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

In [ ]:
def get_embeddings(word_list, model, tokenizer, device, batch_size=32):
    model.eval()
    all_embeddings = []

    for i in tqdm(range(0, len(word_list), batch_size), desc="  Extracting Embeddings", leave=False):
        batch_words = word_list[i : i + batch_size]

        inputs = tokenizer(
            batch_words,
            padding=True,
            truncation=True,
            return_tensors="pt",
            add_special_tokens=True
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            last_hidden_states = outputs.last_hidden_state

        # Mean pooling: average tokens between [CLS] and [SEP]
        batch_emb = []
        lengths = inputs.attention_mask.sum(dim=1)

        for j, seq_len in enumerate(lengths):
            valid_tokens = last_hidden_states[j, 1 : seq_len-1, :]
            if valid_tokens.size(0) == 0:
                word_vec = last_hidden_states[j, 0, :]
            else:
                word_vec = valid_tokens.mean(dim=0)
            batch_emb.append(word_vec.cpu().numpy())

        all_embeddings.extend(batch_emb)

    return np.array(all_embeddings)

In [ ]:
def load_data(file_path):
  with open(file_path, "r", encoding="utf-8") as f:
      data = json.load(f)

  words = []
  labels = []
  for rhyme_ending, word_list in data.items():
      words.extend(word_list)
      labels.extend([rhyme_ending] * len(word_list))

  print(f"Dataset loaded ({file_path}): {len(words)} words covering {len(set(labels))} rhyme classes.")

  le = LabelEncoder()
  y_encoded = le.fit_transform(labels)
  return words, y_encoded

In [ ]:
complete_results = {}
for file_path in FILE_PATHS:
  words, y_encoded = load_data(file_path)
  results = {}

  for model_id in MODELS:
      print(f"\n" + "="*50)
      print(f"Evaluating: {model_id}")
      print("="*50)

      fix_seed(SEED)
      tokenizer = AutoTokenizer.from_pretrained(model_id)
      model = AutoModel.from_pretrained(model_id).to(DEVICE)

      # Extract
      X_embeddings = get_embeddings(words, model, tokenizer, DEVICE, BATCH_SIZE)

      # Split
      X_train, X_test, y_train, y_test = train_test_split(
          X_embeddings, y_encoded, test_size=0.2, random_state=SEED, stratify=y_encoded
      )

      print(len(X_train), len(X_test))

      # Train
      print("  Training Linear Probe...")
      clf = LogisticRegression(max_iter=3000, n_jobs=-1)
      clf.fit(X_train, y_train)

      # Evaluate
      y_pred = clf.predict(X_test)

      # Metrics
      acc = accuracy_score(y_test, y_pred)
      f1 = f1_score(y_test, y_pred, average='macro')

      results[model_id] = {'Accuracy': acc, 'F1': f1}
      print(f"  >> Accuracy: {acc:.4f}")
      print(f"  >> F1 Score: {f1:.4f}")

      del model
      del tokenizer
      torch.cuda.empty_cache()

      complete_results[file_path] = results

In [ ]:
for file_path in FILE_PATHS:
  results = complete_results[file_path]
  print("\n" + "="*80)
  print(f"Final Results for {file_path}:")
  print("="*100)
  print(f"{'Model':<60} | {'Acc':<10} | {'F1':<10}")
  print("-" * 80)
  sorted_results = sorted(results.items(), key=lambda item: item[1]['F1'], reverse=True)

  for model_name, metrics in sorted_results:
      print(f"{model_name:<60} | {metrics['Accuracy']:.4f}   | {metrics['F1']:.4f}")